# 02 · Data cleaning walkthrough (M1 + M2)
Shows each defect in the **raw** files and how the pipeline in `src/clean.py` repairs it. Every step is logged in `reports/data_quality.md`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))          # lets the notebook import the project's src/ package
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

In [ ]:
raw = pd.read_csv(ROOT/"data/raw/traffic_sensor_log.csv")
w = pd.read_csv(ROOT/"data/raw/weather_observations.csv")
print("raw traffic rows:", len(raw))
raw.isna().sum()[lambda s: s>0]

### Defect 1 – duplicates (same road + date + time)

In [ ]:
print("duplicate rows:", raw.duplicated(["road_id","date","time"]).sum())

### Defect 2 – impossible values
Negative volumes, speeds above 200 km/h, occupancy above 100 %. **Insight:** `vehicle_count` still holds the true detector count on rows where `traffic_volume` is null or negative, so those can be recovered *exactly* instead of guessed.

In [ ]:
print("negative volume:", (raw.traffic_volume<0).sum(), "| speed>200:", (raw.avg_speed>200).sum(), "| occupancy>100:", (raw.occupancy>100).sum())
raw.loc[raw.traffic_volume.isna() | (raw.traffic_volume<0), ["traffic_volume","vehicle_count"]].head()

### Defect 3 – inconsistent weather labels and a different date format

In [ ]:
print(w.weather_condition.value_counts())
print(w.date.head(3).tolist(), "(DD/MM/YYYY)  vs traffic", raw.date.head(3).tolist(), "(YYYY-MM-DD)")

### Run the cleaning pipeline

In [ ]:
from src import clean
clean_df = clean.run()
print(clean_df.shape)

In [ ]:
Markdown((ROOT/"reports/data_quality.md").read_text())

### Verify the result
No nulls in the modelling columns, no duplicates, a complete 30-minute grid for every road, weather labels in a controlled vocabulary.

In [ ]:
print(clean_df[["traffic_volume","avg_speed","occupancy","congestion_level","visibility"]].isna().sum().sum(), "nulls")
print(clean_df.weather_condition.value_counts())
print(len(clean_df), "=", clean_df.road_id.nunique(), "roads x", clean_df.timestamp.nunique(), "windows")